In [2]:
from pathlib import Path 
import os 
import warnings 

import numpy as np 
import pandas as pd 
import polars as pl 
import datetime as dt
from datasets import load_dataset 

os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"


c:\Users\Anigma PC\miniconda3\envs\btc_research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
#Load Data from HF 

markets = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "markets"
)
prices = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "prices"
)

ticks = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "ticks"
)

spot_prices = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "spot_prices"
)

"""
orderbook = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "orderbook",
    streaming=True 
)"""



'\norderbook = load_dataset(\n    "aliplayer1/polymarket-crypto-updown",\n    "orderbook",\n    streaming=True \n)'

In [11]:

datasets = { 
    "../data/raw/markets_raw.parquet": markets,
    "../data/raw/prices_raw.parquet": prices,
    "../data/raw/ticks_raw.parquet": ticks,
    "../data/raw/spot_prices_raw.parquet": spot_prices
}

for filepath, dataset in datasets.items():
    if not os.path.exists(filepath):
        print(f"File missing. Processing and saving: {filepath}")
        df = dataset['train'].to_pandas()
        df.to_parquet(filepath, index=False)
    else:
        print(f"File already exists, skipping: {filepath}")


"""
df_markets_raw = markets['train'].to_pandas() 
df_markets_raw.to_parquet("../data/raw/markets_raw.parquet",index=False)

df_prices_raw = prices['train'].to_pandas() 
df_prices_raw.to_parquet("../data/raw/prices_raw.parquet",index=False)

df_ticks_raw = ticks['train'].to_pandas() 
df_ticks_raw.to_parquet("../data/raw/ticks_raw.parquet",index=False)

df_spot_prices_raw = spot_prices['train'].to_pandas()
df_spot_prices_raw.to_parquet("../data/raw/spot_prices_raw.parquet",index=False)
"""

File already exists, skipping: ../data/raw/markets_raw.parquet
File already exists, skipping: ../data/raw/prices_raw.parquet
File already exists, skipping: ../data/raw/ticks_raw.parquet
File missing. Processing and saving: ../data/raw/spot_prices_raw.parquet


'\ndf_markets_raw = markets[\'train\'].to_pandas() \ndf_markets_raw.to_parquet("../data/raw/markets_raw.parquet",index=False)\n\ndf_prices_raw = prices[\'train\'].to_pandas() \ndf_prices_raw.to_parquet("../data/raw/prices_raw.parquet",index=False)\n\ndf_ticks_raw = ticks[\'train\'].to_pandas() \ndf_ticks_raw.to_parquet("../data/raw/ticks_raw.parquet",index=False)\n\ndf_spot_prices_raw = spot_prices[\'train\'].to_pandas()\ndf_spot_prices_raw.to_parquet("../data/raw/spot_prices_raw.parquet",index=False)\n'

In [4]:
# Load Files 

df_markets = pl.scan_parquet("../data/raw/markets_raw.parquet")
df_prices = pl.scan_parquet("../data/raw/prices_raw.parquet")
df_ticks = pl.scan_parquet("../data/raw/ticks_raw.parquet")
df_spot_prices = pl.scan_parquet("../data/raw/spot_prices_raw.parquet")

In [ ]:
## Quick Date Diagnostic 

dates = (
    df_markets
    .filter((pl.col("crypto") == "BTC") & (pl.col("timeframe") == "5-minute"))
    .select(
        pl.col("start_ts").min().alias("min"),
        pl.col("start_ts").max().alias("max"),
    )
    .collect()
)

print(f"Min: {dt.datetime.fromtimestamp(dates['min'][0], tz=dt.timezone.utc)}")
print(f"Max: {dt.datetime.fromtimestamp(dates['max'][0], tz=dt.timezone.utc)}")

In [12]:
## Filter Data from Markets/Prices/Ticks/Spot to access BTC data (5m) + create columns for prediction_ts  

In [ ]:
## MARKETS 

WINDOW = 300

## start_ts : polymarket market open / created (is before window start)
## window_start_ts : start of 5m resolution window 
## end_ts : end of 5m resolution window 

df_markets_btc5m = (
    df_markets
    .filter((pl.col("timeframe") == "5-minute") & (pl.col("crypto") == "BTC"))
    .with_columns((pl.col("end_ts") - WINDOW).alias("window_start_ts"))
)

## PRICES 

df_prices_btc5m = df_prices.join(
    df_markets_btc5m.select("market_id"), on="market_id", how="semi"
)

## TICKS 

df_ticks_btc5m = df_ticks.join(
    df_markets_btc5m.select("market_id"), on="market_id", how="semi"
)

## SPOT PRICES 

df_spot_prices_btc = df_spot_prices.filter(
    (pl.col("symbol") == "btc/usd") & (pl.col("source") == "chainlink_proxy")
)

In [ ]:
# Filter df_prices_btc5m to only include timestamps for prediction_ts (1m before contract resolution)

T1 = 60 

# prediction_ts for every 5m BTC market 

markets_t1 = df_markets_btc5m.with_columns(
    (pl.col("end_ts") - T1).alias("prediction_ts")
)

# attach market start time to each price observation 
prices_t1 = (
    df_prices_btc5m
    .join(
        markets_t1.select(["market_id", "window_start_ts"]),
        on="market_id",
        how="inner",
    )
    # Keep prices at or after the contract's 5-minute window begins
    .filter(pl.col("timestamp") >= pl.col("window_start_ts"))
)

markets_t1 = markets_t1.sort(["market_id", "prediction_ts"])
prices_t1 = prices_t1.sort(["market_id", "timestamp"])

df_t1 = (
    markets_t1.join_asof(
        prices_t1,
        left_on="prediction_ts",
        right_on="timestamp",
        by="market_id",
        strategy="backward",
    )
    .with_columns(
        (pl.col("prediction_ts") - pl.col("timestamp")).alias("price_age_seconds")
    )
    .collect()
)

print(
    df_t1.select(
        pl.col("price_age_seconds").quantile(q, interpolation="linear").alias(str(q))
        for q in [0.90, 0.95, 0.99, 0.995, 0.999, 1.00]
    )
)

df_t1.head()


In [ ]:
df_btc5m_tf.shape

(56476, 23)